# Data Analysis and Visualization Workflows

This notebook demonstrates comprehensive data analysis and visualization workflows for quantum experiments.

## Workflow Overview
1. Data loading and preprocessing
2. Statistical analysis techniques
3. Visualization best practices
4. Error analysis and propagation
5. Performance metrics calculation
6. Report generation

## 1. Data Loading and Preprocessing

In [ ]:
import leeq
import numpy as np
import pandas as pd
from leeq.chronicle import Chronicle, log_and_record
from leeq.core.elements.built_in.qudit_transmon import TransmonElement
from leeq.theory.fits import *
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

Chronicle().start_log()

shots = np.arange(1, 101)
experiment_data = pd.DataFrame({
    "experiment": np.repeat(["Rabi", "Ramsey"], 50),
    "shot": np.tile(np.arange(50), 2),
    "signal": np.concatenate([
        0.5 + 0.45 * np.sin(np.linspace(0, 2 * np.pi, 50)),
        0.5 + 0.35 * np.exp(-np.linspace(0, 3, 50)) * np.cos(np.linspace(0, 8 * np.pi, 50)),
    ]),
})
experiment_data["signal"] += 0.01 * np.sin(shots)

class ExperimentAnalysisWorkflow:
    def __init__(self, data):
        self.data = data

    def row_count(self):
        return int(self.data.shape[0])


analysis_workflow = ExperimentAnalysisWorkflow(experiment_data)

print("QubitSetup analysis data loaded")
print(f"Workflow rows: {analysis_workflow.row_count()}")
print(experiment_data.head())

## 2. Statistical Analysis Techniques

In [ ]:
summary = experiment_data.groupby("experiment")["signal"].agg(["mean", "std", "count"])
summary["stderr"] = summary["std"] / np.sqrt(summary["count"])
summary["ci95"] = 1.96 * summary["stderr"]

rabi_contrast = experiment_data.query("experiment == 'Rabi'")["signal"].max() - experiment_data.query("experiment == 'Rabi'")["signal"].min()
ramsey_visibility = experiment_data.query("experiment == 'Ramsey'")["signal"].std()

print("Statistical analysis complete")
print(summary)
print(f"Rabi contrast: {rabi_contrast:.4f}")
print(f"Ramsey visibility: {ramsey_visibility:.4f}")

## 3. Visualization Best Practices

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Experiment traces", "Mean signal with CI"))

for name, group in experiment_data.groupby("experiment"):
    fig.add_trace(go.Scatter(x=group["shot"], y=group["signal"], mode="lines", name=name), row=1, col=1)

fig.add_trace(go.Bar(
    x=summary.index,
    y=summary["mean"],
    error_y={"type": "data", "array": summary["ci95"]},
    name="Mean signal",
), row=1, col=2)

fig.update_layout(title="Experiment Analysis Workflow", height=420, showlegend=True)
fig.show()

## 6. Report Generation

In [ ]:
analysis_report = {
    "rabi_contrast": float(rabi_contrast),
    "ramsey_visibility": float(ramsey_visibility),
    "experiments_analyzed": int(summary.shape[0]),
}

log_and_record("experiment_analysis_report", analysis_report)

print("Analysis report")
for key, value in analysis_report.items():
    print(f"  {key}: {value}")